In [ ]:
!pip install ragas[all]
!pip install litellm

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 3.3 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of llama-cloud-services to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of llama-cloud-services to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.9/160.9 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 42.9 MB/s eta 0:00:00
   

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 102.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.9/5.9 MB 70.0 MB/s eta 0:00:00
  Attempting uninstall: grpcio
    Found existing installation: grpcio 1.76.0
    Uninstalling grpcio-1.76.0:
      Successfully uninstalled grpcio-1.76.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.21.0 requires starlette<1.0.0,>=0.49.1, but you have starlette 0.46.2 which is incompatible.
grpcio-status 1.71.2 requires grpcio>=1.71.2, but you have grpcio 1.67.1 which is incompatible.


In [ ]:
# Import libraries and setup config
import os
import pandas as pd
from google.colab import userdata
from openai import AsyncOpenAI
from ragas.llms import llm_factory
from ragas.metrics.collections import ContextPrecision, ContextRecall, Faithfulness, AnswerCorrectness, FactualCorrectness
from ragas.embeddings.base import embedding_factory

# Set the OPENAI_API_KEY environment variable from Colab secrets
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

# Setup LLM
client = AsyncOpenAI()
# Increase max_tokens for the LLM to prevent truncation of structured outputs
llm = llm_factory("gpt-4o-mini", client=client, max_tokens=4096)
embeddings = embedding_factory("openai", model="text-embedding-3-small", client=client)


In [ ]:
# Import data and pre-process

qa_data_raw = pd.read_csv('/content/drive/MyDrive/RAG/datasets/rag_with_gemma3_result_20251220.csv')
qa_data_raw = qa_data_raw[['query', 'answer_ref', 'answer', 'content', 'passage_1_text']]
qa_data_grouped = qa_data_raw.groupby(['query', 'answer_ref', 'answer', 'passage_1_text'])['content'].agg(list).reset_index()
qa_data = qa_data_grouped.rename(columns={'query': 'user_input',
                                                  'passage_1_text': 'reference',
                                                  'content': 'retrieved_contexts'})
# display(qa_data.head())

qa_data_dict = qa_data.to_dict()

qa_data_expanded = qa_data['retrieved_contexts'].apply(lambda x: pd.Series(x + [None]*(4-len(x)) if len(x) < 4 else x[:4]))
qa_data_expanded.columns = [f'retrieved_contexts_{i+1}' for i in range(4)]
qa_data_final = pd.concat([qa_data.drop(columns=['retrieved_contexts']), qa_data_expanded], axis=1)

display(qa_data_final.head())





,user_input,answer_ref,answer,reference,retrieved_contexts_1,retrieved_contexts_2,retrieved_contexts_3,retrieved_contexts_4
0,"""According to the passages, during what time p...","""The second school closure occurred from Decem...",I'd be happy to help you with that!\n\nStep 1:...,"""Then primary/elementary schools were closed o...",FRANCE \nPIRLS 2021 ENCYCLOPEDIA 13 \n \n \nTh...,FRANCE \nPIRLS 2021 ENCYCLOPEDIA 13 \n \n \nTh...,FRANCE \nPIRLS 2021 ENCYCLOPEDIA 13 \n \n \nTh...,FRANCE \nPIRLS 2021 ENCYCLOPEDIA 13 \n \n \nTh...
1,"""According to the passages, how did the COVID-...","""During the COVID-19 pandemic, many educationa...","To answer this question, I will walk you throu...","""Due to disruptions caused by the COVID-19 pan...","student achievement, including NAEP, were post...","student achievement, including NAEP, were post...","student achievement, including NAEP, were post...","student achievement, including NAEP, were post..."
2,"""According to the passages, what constitutiona...","""Freedom of education (guaranteed by the Dutch...",I'd be happy to walk you through my thought pr...,"""One of the key features of the Dutch educatio...",NETHERLANDS \nPIRLS 2021 ENCYCLOPEDIA 1 \n \n ...,NETHERLANDS \nPIRLS 2021 ENCYCLOPEDIA 1 \n \n ...,NETHERLANDS \nPIRLS 2021 ENCYCLOPEDIA 1 \n \n ...,NETHERLANDS \nPIRLS 2021 ENCYCLOPEDIA 1 \n \n ...
3,"""According to the passages, what is the name o...","""The law is the Every Student Succeeds Act (ES...",I'd be happy to walk you through my thought pr...,"""In December 2015, Congress reauthorized ESEA ...","regulations, elected or appointed school board...","regulations, elected or appointed school board...","regulations, elected or appointed school board...","regulations, elected or appointed school board..."
4,"""According to the passages, what specific type...","""The SkillsFuture Work-Study Programs, which p...",I'll walk you through my thought process step ...,"""In addition to providing preemployment traini...",providers of continuing education and training...,providers of continuing education and training...,providers of continuing education and training...,providers of continuing education and training...


## Evaluation

### Context Precision and Recall

In [ ]:
%%time
import asyncio
import nest_asyncio

context_precision_scorer = ContextPrecision(llm=llm)
context_recall_scorer = ContextRecall(llm=llm)

context_precision_scores = []
context_recall_scores = []

async def calculate_scores():
    for index, row in qa_data.iterrows():
        context_precision_result = await context_precision_scorer.ascore(
            user_input=row['user_input'],
            reference=row['reference'],
            retrieved_contexts=row['retrieved_contexts'],
        )

        context_precision_scores.append(context_precision_result.value)

        context_recall_result = await context_recall_scorer.ascore(
            user_input=row['user_input'],
            reference=row['reference'],
            retrieved_contexts=row['retrieved_contexts'],
        )

        context_recall_scores.append(context_recall_result.value)

nest_asyncio.apply()
asyncio.run(calculate_scores())

qa_data['context_precision_score'] = context_precision_scores
qa_data['context_recall_score'] = context_recall_scores

CPU times: user 2min 5s, sys: 2.01 s, total: 2min 7s
Wall time: 45min 28s


In [ ]:
# Save result to CSV
qa_data.to_csv('/content/drive/MyDrive/RAG/results/eval_precision_recall_gemma_20251222.csv', index=False)

### Faithfulness

In [ ]:
%%time
import asyncio
import nest_asyncio

faithfulness_scorer = Faithfulness(llm=llm)
faithfulness_scores = []

async def calculate_faithfulness():
    for index, row in qa_data.iterrows():
        faithfulness_result = await faithfulness_scorer.ascore(
            user_input=row['user_input'],
            response=row['answer'],
            retrieved_contexts=row['retrieved_contexts'],
        )

        faithfulness_scores.append(faithfulness_result.value)

nest_asyncio.apply()
asyncio.run(calculate_faithfulness())

qa_data['faithfulness_score'] = faithfulness_scores



In [ ]:
qa_data.columns

In [ ]:
# Save result to CSV
qa_data.to_csv('/content/drive/MyDrive/RAG/results/eval_with_faithfulness_gemma_20251222.csv', index=False)


### Answer Correctness

In [ ]:
%%time
import asyncio
import nest_asyncio

answer_correctness_scorer = AnswerCorrectness(llm=llm, embeddings=embeddings)
answer_correctness_scores = []

async def calculate_answer_correctness():
    for index, row in qa_data.iterrows():
        answer_correctness_result = await answer_correctness_scorer.ascore(
            user_input=row['user_input'],
            response=row['answer'],
            reference=row['reference'],
        )

        answer_correctness_scores.append(answer_correctness_result.value)

nest_asyncio.apply()
asyncio.run(calculate_answer_correctness())

qa_data['answer_correctness_score'] = answer_correctness_scores

In [ ]:
# Save result to CSV
qa_data.to_csv('/content/drive/MyDrive/RAG/results/eval_with_answer_correctness_gemma_20251222.csv', index=False)

### Factual Correctness

In [ ]:
%%time
import asyncio
import nest_asyncio

factual_correctness_scorer = FactualCorrectness(llm=llm)
factual_correctness_scores = []

async def calculate_factual_correctness():
    for index, row in qa_data.iterrows():
        factual_correctness_result = await factual_correctness_scorer.ascore(
            response=row['answer'],
            reference=row['reference'],
            )

        factual_correctness_scores.append(factual_correctness_result.value)

nest_asyncio.apply()
asyncio.run(calculate_factual_correctness())

qa_data['factual_correctness_score'] = factual_correctness_scores

In [ ]:
# Save result to CSV
qa_data.to_csv('/content/drive/MyDrive/RAG/results/eval_with_factual_correctness_gemma_20251222.csv', index=False)